# 02 — Structured Signals (Signal A)

Builds structured early-warning candidates from ASRS multi-label fields, following `production/spec/P03_methodology.md` (§3–§5, §9).

Outputs land in `production/output/signals/`:
- `signal_candidates.csv`
- `category_share_series.parquet`
- `P04_signalA_summary.md`
- top-candidate PNG trend plots

In [1]:
# provenance: primary=OP48 authors=[OP48, CDX53]
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
import math

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "asrs.parquet").exists():
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / "data" / "processed" / "asrs.parquet"
OUT_DIR = ROOT / "production" / "output" / "signals"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_START = 201101
WINDOW_END = 202512
TRAILING_EXCLUDE_MONTHS = 2
MIN_AVG_MONTHLY_COUNT = 20
BASELINE_MONTHS = 24
WATCH_Z = 2.0
STRONG_Z = 3.0
PERSISTENCE_MONTHS = 3

FIELDS = [
    "Anomaly",
    "Contributing Factors / Situations",
    "Person 1 | Human Factors",
    "Aircraft 1 | Flight Phase",
]

print(f"DATA_PATH={DATA_PATH}")
print(f"OUT_DIR={OUT_DIR}")

DATA_PATH=data/processed/asrs.parquet
OUT_DIR=production/output/signals


In [2]:
def ym_to_datetime(ym: pd.Series) -> pd.Series:
    return pd.to_datetime(ym.astype(str), format="%Y%m")


def rolling_mad(values: np.ndarray) -> float:
    med = np.median(values)
    return float(np.median(np.abs(values - med)))


def longest_run(mask: pd.Series) -> int:
    run = 0
    best = 0
    for v in mask.fillna(False).astype(bool):
        if v:
            run += 1
            best = max(best, run)
        else:
            run = 0
    return best


def first_run_ym(mask: pd.Series, yms: pd.Series, min_run: int) -> float | np.nan:
    run = 0
    for ok, ym in zip(mask.fillna(False).astype(bool), yms):
        if ok:
            run += 1
            if run >= min_run:
                return ym
        else:
            run = 0
    return np.nan


def build_calendar(start_ym: int, end_ym: int) -> pd.DataFrame:
    months = pd.period_range(str(start_ym), str(end_ym), freq="M")
    ym = pd.Series([int(p.strftime("%Y%m")) for p in months], name="ym")
    return pd.DataFrame({"ym": ym, "date": pd.to_datetime(ym.astype(str), format="%Y%m")})


calendar = build_calendar(WINDOW_START, WINDOW_END)
calendar.head()

,ym,date
0,201101,2011-01-01
1,201102,2011-02-01
2,201103,2011-03-01
3,201104,2011-04-01
4,201105,2011-05-01


In [3]:
# Load and window base data.
df_pl = (
    pl.read_parquet(DATA_PATH)
    .with_columns(
        pl.col("Date").cast(pl.Utf8).str.slice(0, 6).alias("ym_str"),
        pl.col("Date").cast(pl.Int64).alias("ym"),
    )
    .filter((pl.col("ym") >= WINDOW_START) & (pl.col("ym") <= WINDOW_END))
)

base_pdf = df_pl.select(["ACN", "ym", *FIELDS]).to_pandas()

# Total monthly report volume.
totals = (
    df_pl.group_by("ym")
    .len()
    .rename({"len": "total_reports"})
    .sort("ym")
    .to_pandas()
)

calendar_totals = calendar.merge(totals, on="ym", how="left")
calendar_totals["total_reports"] = calendar_totals["total_reports"].fillna(0).astype(int)

print("rows", len(base_pdf), "cols", base_pdf.shape[1])
print("months", len(calendar_totals), "range", calendar_totals["ym"].min(), "->", calendar_totals["ym"].max())

rows 79577 cols 6
months 180 range 201101 -> 202512


In [4]:
# Explode multi-label fields and compute monthly count/share per category.
frames: list[pd.DataFrame] = []

for field in FIELDS:
    tmp = base_pdf[["ACN", "ym", field]].copy()
    tmp[field] = tmp[field].fillna("").astype(str)
    tmp["category"] = tmp[field].str.split("; ")
    tmp = tmp.explode("category")
    tmp["category"] = tmp["category"].fillna("").astype(str).str.strip()
    tmp = tmp[tmp["category"] != ""]
    tmp["field"] = field
    g = tmp.groupby(["field", "category", "ym"], as_index=False).size()
    g = g.rename(columns={"size": "count"})
    frames.append(g)

counts = pd.concat(frames, ignore_index=True)
series = counts.merge(calendar_totals[["ym", "date", "total_reports"]], on="ym", how="left")
series["share"] = np.where(series["total_reports"] > 0, series["count"] / series["total_reports"], np.nan)

# Build complete monthly series per (field, category) so rolling stats are valid.
all_groups = series[["field", "category"]].drop_duplicates()
all_rows = all_groups.merge(calendar_totals[["ym", "date", "total_reports"]], how="cross")
full = all_rows.merge(series[["field", "category", "ym", "count", "share"]], on=["field", "category", "ym"], how="left")
full["count"] = full["count"].fillna(0).astype(int)
full["share"] = np.where(full["total_reports"] > 0, full["count"] / full["total_reports"], np.nan)

# Min-volume floor.
mean_counts = full.groupby(["field", "category"], as_index=False)["count"].mean().rename(columns={"count": "avg_count_per_month"})
keepers = mean_counts[mean_counts["avg_count_per_month"] >= MIN_AVG_MONTHLY_COUNT][["field", "category"]]
full = full.merge(keepers, on=["field", "category"], how="inner")

print("category families before floor:", len(all_groups))
print("category families after floor:", len(keepers))
full.head()

category families before floor: 339
category families after floor: 42


,field,category,ym,date,total_reports,count,share
0,Anomaly,ATC Issue All Types,201101,2011-01-01,406,98,0.241379
1,Anomaly,ATC Issue All Types,201102,2011-02-01,375,72,0.192000
2,Anomaly,ATC Issue All Types,201103,2011-03-01,377,111,0.294430
3,Anomaly,ATC Issue All Types,201104,2011-04-01,473,128,0.270613
4,Anomaly,ATC Issue All Types,201105,2011-05-01,465,97,0.208602


In [5]:
# Signal scoring per category family.
last_ym = int(calendar_totals["ym"].max())
last_non_claimable = set(calendar_totals.sort_values("ym").tail(TRAILING_EXCLUDE_MONTHS)["ym"].tolist())

rows = []
scored_frames = []

for (field, category), g in full.groupby(["field", "category"], sort=False):
    g = g.sort_values("ym").copy()

    # Baseline from trailing 24 months, excluding current month.
    base_series = g["share"].shift(1)
    g["median_24"] = base_series.rolling(BASELINE_MONTHS, min_periods=BASELINE_MONTHS).median()
    g["mad_24"] = base_series.rolling(BASELINE_MONTHS, min_periods=BASELINE_MONTHS).apply(rolling_mad, raw=True)

    denom = 1.4826 * g["mad_24"]
    g["z"] = np.where(denom > 0, (g["share"] - g["median_24"]) / denom, np.nan)
    g["z"] = g["z"].replace([np.inf, -np.inf], np.nan)

    # Year-over-year seasonality guard.
    g["yoy_share_delta"] = g["share"] - g["share"].shift(12)

    # Taxonomy guard: category first appears after analysis start.
    nz = g.loc[g["count"] > 0, "ym"]
    first_nonzero_ym = int(nz.iloc[0]) if len(nz) else np.nan
    taxonomy_suspect = bool(first_nonzero_ym > WINDOW_START) if not pd.isna(first_nonzero_ym) else True

    g["is_trailing_excluded"] = g["ym"].isin(last_non_claimable)
    baseline_ready = g["median_24"].notna() & g["mad_24"].notna()
    common_fire = baseline_ready & (~g["is_trailing_excluded"]) & (g["yoy_share_delta"] > 0)

    g["watch_fire"] = common_fire & (g["z"] >= WATCH_Z)
    g["strong_fire"] = common_fire & (g["z"] >= STRONG_Z)

    watch_run_len = longest_run(g["watch_fire"])
    strong_run_len = longest_run(g["strong_fire"])

    first_watch = first_run_ym(g["watch_fire"], g["ym"], PERSISTENCE_MONTHS)
    first_strong = first_run_ym(g["strong_fire"], g["ym"], PERSISTENCE_MONTHS)

    if not pd.isna(first_strong):
        first_fire_ym = int(first_strong)
        fire_level = "strong"
        months_sustained = strong_run_len
    elif not pd.isna(first_watch):
        first_fire_ym = int(first_watch)
        fire_level = "watch"
        months_sustained = watch_run_len
    else:
        first_fire_ym = np.nan
        fire_level = "none"
        months_sustained = 0

    latest = g.iloc[-1]
    peak_z = float(g.loc[g["watch_fire"], "z"].max()) if g["watch_fire"].any() else np.nan

    rows.append(
        {
            "field": field,
            "category": category,
            "latest_share": float(latest["share"]) if pd.notna(latest["share"]) else np.nan,
            "z_latest": float(latest["z"]) if pd.notna(latest["z"]) else np.nan,
            "yoy_share_delta": float(latest["yoy_share_delta"]) if pd.notna(latest["yoy_share_delta"]) else np.nan,
            "taxonomy_suspect": taxonomy_suspect,
            "first_nonzero_ym": first_nonzero_ym,
            "first_fire_ym": first_fire_ym,
            "fire_level": fire_level,
            "months_sustained": int(months_sustained),
            "peak_z": peak_z,
            "rank": (float(peak_z) * int(months_sustained)) if pd.notna(peak_z) else 0.0,
        }
    )

    scored_frames.append(g.assign(field=field, category=category))

candidates = pd.DataFrame(rows)
scored = pd.concat(scored_frames, ignore_index=True)

# Keep ranked candidates for human review, not alarms.
candidates = candidates.sort_values(["rank", "peak_z"], ascending=False).reset_index(drop=True)

print("candidate rows", len(candidates))
candidates.head(15)

candidate rows 42


,field,category,latest_share,z_latest,yoy_share_delta,taxonomy_suspect,first_nonzero_ym,first_fire_ym,fire_level,months_sustained,peak_z,rank
0,Person 1 | Human Factors,Troubleshooting,0.161702,-0.916023,-0.052921,False,201101,202109.0,strong,11,9.133662,100.470283
1,Contributing Factors / Situations,Environment - Non Weather Related,0.123404,0.477042,0.043216,False,201101,202005.0,strong,11,8.768607,96.454675
2,Anomaly,Flight Deck / Cabin / Aircraft Event Smoke / F...,0.038298,0.792582,0.000562,False,201101,201905.0,strong,6,14.387037,86.322220
3,Anomaly,Conflict Airborne Conflict,0.023404,-0.595945,-0.011973,False,201101,201405.0,strong,6,7.727871,46.367228
4,Anomaly,Deviation / Discrepancy - Procedural Published...,0.408511,-1.590320,0.000492,False,201101,201403.0,strong,5,6.806917,34.034587
5,Person 1 | Human Factors,Situational Awareness,0.359574,-0.914027,-0.005992,False,201101,201907.0,strong,4,7.186465,28.745860
6,Anomaly,Inflight Event / Encounter CFTT / CFIT,0.114894,0.013617,0.008762,False,201101,201401.0,strong,3,9.535270,28.605809
7,Anomaly,Deviation / Discrepancy - Procedural FAR,0.072340,-0.956110,-0.010207,False,201101,201508.0,strong,4,6.757066,27.028265
8,Person 1 | Human Factors,Human-Machine Interface,0.051064,0.266541,0.018045,False,201101,201711.0,watch,3,8.886229,26.658686
9,Person 1 | Human Factors,Confusion,0.061702,-1.082707,-0.034996,False,201101,201406.0,strong,4,6.461337,25.845348


In [6]:
# Persist artifacts.
series_out = scored[
    [
        "field",
        "category",
        "ym",
        "date",
        "count",
        "total_reports",
        "share",
        "median_24",
        "mad_24",
        "z",
        "yoy_share_delta",
        "is_trailing_excluded",
        "watch_fire",
        "strong_fire",
    ]
].copy()

series_out_pl = pl.from_pandas(series_out)
series_out_pl.write_parquet(OUT_DIR / "category_share_series.parquet")

candidate_cols = [
    "field",
    "category",
    "latest_share",
    "z_latest",
    "yoy_share_delta",
    "taxonomy_suspect",
    "first_fire_ym",
    "months_sustained",
    "peak_z",
    "rank",
    "fire_level",
]
candidates[candidate_cols].to_csv(OUT_DIR / "signal_candidates.csv", index=False)

# Quick sanity: multi-label means sums can exceed 1, but should be >=1 where field has values.
share_sums = (
    series_out.groupby(["field", "ym"], as_index=False)["share"].sum().rename(columns={"share": "share_sum"})
)
print("share_sums min/max:")
print(share_sums.groupby("field")["share_sum"].agg(["min", "max"]).round(3))

print("wrote", OUT_DIR / "category_share_series.parquet")
print("wrote", OUT_DIR / "signal_candidates.csv")

share_sums min/max:
                                     min    max
field                                          
Aircraft 1 | Flight Phase          0.895  1.070
Anomaly                            1.490  2.771
Contributing Factors / Situations  1.624  2.332
Person 1 | Human Factors           1.105  2.592
wrote production/output/signals/category_share_series.parquet
wrote production/output/signals/signal_candidates.csv


In [7]:
# Plot top ranked non-taxonomy-suspect candidates.
plot_dir = OUT_DIR / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

top = candidates[
    (~candidates["taxonomy_suspect"]) & (candidates["first_fire_ym"].notna())
].head(6)

for _, row in top.iterrows():
    field = row["field"]
    category = row["category"]

    g = scored[(scored["field"] == field) & (scored["category"] == category)].sort_values("ym")

    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(g["date"], g["share"], label="Share", linewidth=1.6)
    ax.plot(g["date"], g["median_24"], label="24m baseline (median)", linewidth=1.2)

    band = 1.4826 * g["mad_24"]
    lower = g["median_24"] - band
    upper = g["median_24"] + band
    ax.fill_between(g["date"], lower, upper, alpha=0.15, label="Baseline band (±1 MAD)")

    watch = g[g["watch_fire"]]
    strong = g[g["strong_fire"]]
    if not watch.empty:
        ax.scatter(watch["date"], watch["share"], s=18, label="Watch fire", zorder=3)
    if not strong.empty:
        ax.scatter(strong["date"], strong["share"], s=24, label="Strong fire", zorder=4)

    if TRAILING_EXCLUDE_MONTHS > 0:
        trail = g[g["is_trailing_excluded"]]
        if not trail.empty:
            start = trail["date"].min()
            end = trail["date"].max() + pd.offsets.MonthEnd(1)
            ax.axvspan(start, end, color="gray", alpha=0.12, label="Trailing months (excluded)")

    ax.set_title(f"{field} — {category}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Share of monthly reports")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=0)
    ax.margins(x=0.01)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()

    safe_name = f"{field}__{category}".replace("/", "-").replace("|", "-").replace(" ", "_")
    safe_name = "".join(ch for ch in safe_name if ch.isalnum() or ch in {"_", "-"})
    out_png = plot_dir / f"trend_{safe_name[:100]}.png"
    fig.savefig(out_png, dpi=140)
    plt.close(fig)

print("plot files:", len(list(plot_dir.glob("trend_*.png"))))

summary_lines = []
summary_lines.append("# Signal A summary\n")
summary_lines.append("Structured outputs are ranked **candidates for human review**, not automated alarms.\n")
summary_lines.append(f"- Candidate rows: **{len(candidates)}**\n")
summary_lines.append(f"- Candidates with a persisted fire (watch/strong): **{int(candidates['first_fire_ym'].notna().sum())}**\n")
summary_lines.append(f"- Taxonomy-suspect categories (gated): **{int(candidates['taxonomy_suspect'].sum())}**\n")
summary_lines.append(f"- Top plotted candidates (non-taxonomy-suspect): **{len(top)}**\n")

if len(top):
    summary_lines.append("\n## Top candidates\n")
    for _, r in top.iterrows():
        summary_lines.append(
            f"- `{r['field']}` → `{r['category']}` | first fire `{int(r['first_fire_ym'])}` | "
            f"level `{r['fire_level']}` | peak z `{r['peak_z']:.2f}` | rank `{r['rank']:.2f}`\n"
        )

(OUT_DIR / "P04_signalA_summary.md").write_text("".join(summary_lines), encoding="utf-8")
print("wrote", OUT_DIR / "P04_signalA_summary.md")

plot files: 6
wrote production/output/signals/P04_signalA_summary.md


In [8]:
candidates.head(20)

,field,category,latest_share,z_latest,yoy_share_delta,taxonomy_suspect,first_nonzero_ym,first_fire_ym,fire_level,months_sustained,peak_z,rank
0,Person 1 | Human Factors,Troubleshooting,0.161702,-0.916023,-0.052921,False,201101,202109.0,strong,11,9.133662,100.470283
1,Contributing Factors / Situations,Environment - Non Weather Related,0.123404,0.477042,0.043216,False,201101,202005.0,strong,11,8.768607,96.454675
2,Anomaly,Flight Deck / Cabin / Aircraft Event Smoke / F...,0.038298,0.792582,0.000562,False,201101,201905.0,strong,6,14.387037,86.322220
3,Anomaly,Conflict Airborne Conflict,0.023404,-0.595945,-0.011973,False,201101,201405.0,strong,6,7.727871,46.367228
4,Anomaly,Deviation / Discrepancy - Procedural Published...,0.408511,-1.590320,0.000492,False,201101,201403.0,strong,5,6.806917,34.034587
5,Person 1 | Human Factors,Situational Awareness,0.359574,-0.914027,-0.005992,False,201101,201907.0,strong,4,7.186465,28.745860
6,Anomaly,Inflight Event / Encounter CFTT / CFIT,0.114894,0.013617,0.008762,False,201101,201401.0,strong,3,9.535270,28.605809
7,Anomaly,Deviation / Discrepancy - Procedural FAR,0.072340,-0.956110,-0.010207,False,201101,201508.0,strong,4,6.757066,27.028265
8,Person 1 | Human Factors,Human-Machine Interface,0.051064,0.266541,0.018045,False,201101,201711.0,watch,3,8.886229,26.658686
9,Person 1 | Human Factors,Confusion,0.061702,-1.082707,-0.034996,False,201101,201406.0,strong,4,6.461337,25.845348
